# House Prices: Advanced Regression Techniques
**Pipeline**: 前処理 → SHAP Top-40 → **2-Stage Residual Learning** → マルチシード提出

- Public Best: **0.11993**
- **Neighborhood_te 廃止** → 地理的特徴量 (Lat/Lon/Dist_to_Center/Dist_to_HighPrice_Center/KNN Geo Price) に移行
- **Stage 1**: Ridge baseline (TotalSF, OverallQual, LotArea) — 基本的な価値
- **Stage 2**: 7モデル OOF スタッキング on 残差 — 付加価値の学習
- ドメイン特徴量 + QOL_Score
- マルチシード OOF スタッキング (seeds: 42, 123, 456)

## Setup (Colab / Local 自動検出)

In [ ]:
import sys, shutil

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in __import__('os').environ
HAS_GPU = shutil.which('nvidia-smi') is not None
print(f"Environment: {'Colab' if IS_COLAB else 'Local'}, GPU: {HAS_GPU}")

if HAS_GPU:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    for line in result.stdout.split('\n')[:4]:
        print(line)

if IS_COLAB:
    !pip install -q lightgbm xgboost
    !pip install -q catboost --upgrade

In [ ]:
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os, json, zipfile
from pathlib import Path

# データパス: Local なら house-prices/data/, Colab なら ./data/
if IS_COLAB:
    DATA_DIR = Path("data")
else:
    candidates = [Path("data"), Path("house-prices/data")]
    try:
        candidates.insert(0, Path(__vsc_ipynb_file__).parent / "data")
    except NameError:
        pass
    DATA_DIR = Path("data")
    for candidate in candidates:
        if (candidate / "train.csv").exists():
            DATA_DIR = candidate
            break

print(f"CWD: {os.getcwd()}")
print(f"DATA_DIR: {DATA_DIR.resolve()}")
DATA_DIR.mkdir(exist_ok=True)

if not (DATA_DIR / "train.csv").exists():
    # === 方法1: Google Drive にデータがあればコピー ===
    if IS_COLAB:
        drive_data = Path("/content/drive/MyDrive/kaggle-data/house-prices")
        if (drive_data / "train.csv").exists():
            import shutil
            for f in ["train.csv", "test.csv", "data_description.txt", "sample_submission.csv"]:
                src = drive_data / f
                if src.exists():
                    shutil.copy2(str(src), str(DATA_DIR / f))
            print(f"Copied data from Google Drive: {drive_data}")

    # === 方法2: kaggle CLI でダウンロード ===
    if not (DATA_DIR / "train.csv").exists():
        # kaggle.json を ~/.kaggle/ に配置
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_json_path = kaggle_dir / "kaggle.json"

        if not kaggle_json_path.exists():
            creds = None

            if IS_COLAB:
                try:
                    from google.colab import userdata
                    raw = userdata.get('KAGGLE_TOKEN')
                    if raw:
                        try:
                            creds = json.loads(raw)
                        except (json.JSONDecodeError, TypeError):
                            try:
                                u = userdata.get('KAGGLE_USERNAME')
                                creds = {"username": u, "key": raw}
                            except Exception:
                                pass
                except Exception:
                    pass

            if creds is None and IS_COLAB:
                drive_kaggle = Path("/content/drive/MyDrive/.kaggle/kaggle.json")
                if drive_kaggle.exists():
                    creds = json.loads(drive_kaggle.read_text())
                    print("Loaded kaggle.json from Google Drive")

            if creds is None:
                raise RuntimeError(
                    "Data not found. Options:\n"
                    "  1. Upload train.csv & test.csv to Google Drive:\n"
                    "     MyDrive/kaggle-data/house-prices/train.csv\n"
                    "     MyDrive/kaggle-data/house-prices/test.csv\n"
                    '  2. Set Colab Secret \'KAGGLE_TOKEN\' as JSON: {"username":"...","key":"..."}\n'
                    "  3. Place kaggle.json at Google Drive: MyDrive/.kaggle/kaggle.json"
                )

            kaggle_dir.mkdir(exist_ok=True)
            kaggle_json_path.write_text(json.dumps(creds))
            os.chmod(str(kaggle_json_path), 0o600)

        print("Downloading via kaggle CLI...")
        !pip install -q --upgrade kaggle
        !kaggle competitions download -c house-prices-advanced-regression-techniques -p {DATA_DIR}

        zip_path = DATA_DIR / "house-prices-advanced-regression-techniques.zip"
        if zip_path.exists():
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(DATA_DIR)
            zip_path.unlink()
            print("Data ready!")
        elif not (DATA_DIR / "train.csv").exists():
            raise RuntimeError(
                "Kaggle download failed (token expired?).\n"
                "Easiest fix: upload train.csv & test.csv to Google Drive:\n"
                "  MyDrive/kaggle-data/house-prices/"
            )
else:
    print(f"Data already exists at {DATA_DIR}")

# 以降のセルで使うパス
TRAIN_CSV = str(DATA_DIR / "train.csv")
TEST_CSV = str(DATA_DIR / "test.csv")
print(f"TRAIN_CSV: {TRAIN_CSV}\nTEST_CSV: {TEST_CSV}")

## Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

## 前処理 (Preprocessor)

In [ ]:
# 合成特徴量に集約済みのため削除する元変数
REDUNDANT_COLS = [
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea',  # → TotalSF
    'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath',  # → TotalBath
    'YearBuilt',                                       # → HouseAge
    'GarageArea',                                      # GarageCars と高相関 → GarageCars を残す
]

# SHAP Top 40: 表現力と防御力の両立
SHAP_TOP_N = 40

# 予測値の物理的クリッピング (訓練データ分布に基づく)
CLIP_MIN = 35000
CLIP_MAX = 600000

# --- 近隣エリア代表座標 (AmesHousing R package + GIS data) ---
NEIGHBORHOOD_COORDS = {
    "Blmngtn": (42.062736, -93.641598),
    "Blueste": (42.055300, -93.648000),
    "BrDale":  (42.052066, -93.628496),
    "BrkSide": (42.032398, -93.624985),
    "ClearCr": (42.028498, -93.671138),
    "CollgCr": (42.018929, -93.686690),
    "Crawfor": (42.015566, -93.641049),
    "Edwards": (42.022096, -93.666631),
    "Gilbert": (42.059746, -93.639344),
    "Greens":  (42.042805, -93.648595),
    "GrnHill": (42.045800, -93.636000),
    "IDOTRR":  (42.022610, -93.623342),
    "Landmrk": (42.021813, -93.682339),
    "MeadowV": (41.991967, -93.603282),
    "Mitchel": (41.991899, -93.604923),
    "NAmes":   (42.042436, -93.617476),
    "NPkVill": (42.050346, -93.625925),
    "NWAmes":  (42.049561, -93.634022),
    "NoRidge": (42.051411, -93.653381),
    "NridgHt": (42.060132, -93.653213),
    "OldTown": (42.031146, -93.614256),
    "SWISU":   (42.020022, -93.651120),
    "Sawyer":  (42.033277, -93.670478),
    "SawyerW": (42.034593, -93.686533),
    "Somerst": (42.049815, -93.644198),
    "StoneBr": (42.060198, -93.631873),
    "Timber":  (41.996842, -93.648400),
    "Veenker": (42.041435, -93.651866),
}
# 高価格エリア中心 (NridgHt, NoRidge, StoneBr の重心)
HIGH_PRICE_CENTER = (
    np.mean([42.060132, 42.051411, 42.060198]),
    np.mean([-93.653213, -93.653381, -93.631873])
)
# Ames 市中心
AMES_CENTER = (42.05, -93.65)


class HousePricesPreprocessor(BaseEstimator, TransformerMixin):
    """前処理: Neighborhood_te を廃止し、地理的特徴量 (Lat/Lon/Distance) に移行"""

    NA_MEANS_NONE = [
        'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
        'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
        'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
        'MasVnrType',
    ]

    ORDINAL_MAP = {
        'ExterQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'ExterCond': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
        'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'HeatingQC': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
        'FireplaceQu': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
        'GarageQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'PavedDrive': {'N': 0, 'P': 1, 'Y': 2},
        'PoolQC': {'None': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4},
        'Fence': {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
        'CentralAir': {'N': 0, 'Y': 1},
        'Street': {'Grvl': 0, 'Pave': 1},
        'LandSlope': {'Sev': 0, 'Mod': 1, 'Gtl': 2},
        'LotShape': {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3},
    }

    DROP_COLS = ['Id', 'Utilities', 'Street']

    SKEW_FEATURES = [
        'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
        'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
        'GrLivArea', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
        'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal',
    ]

    # TE 廃止: 空リスト
    TE_COLS = []
    TE_SMOOTH = 20

    def __init__(self, selected_features=None):
        self.selected_features = selected_features
        self.lot_frontage_medians_ = None
        self.lot_frontage_global_ = None
        self.num_medians_ = None
        self.cat_modes_ = None
        self.feature_names_ = None
        self.cat_feature_names_ = None
        self.cat_feature_indices_ = None
        self.te_stats_ = {}
        self.te_global_mean_ = None
        self._train_y_ = None
        self.first_floor_median_ = None

    def fit(self, X, y=None):
        df = X.copy()
        self.lot_frontage_medians_ = df.groupby('Neighborhood')['LotFrontage'].median()
        self.lot_frontage_global_ = df['LotFrontage'].median()
        num_cols = df.select_dtypes(include=[np.number]).columns
        self.num_medians_ = df[num_cols].median()
        cat_cols = df.select_dtypes(include=['object']).columns
        self.cat_modes_ = df[cat_cols].mode().iloc[0]
        if '1stFlrSF' in df.columns:
            self.first_floor_median_ = df['1stFlrSF'].median()
        if y is not None:
            temp = self._fill_missing(df.copy())
            self._fit_te(temp, y)
        return self

    def fit_transform(self, X, y=None, **fit_params):
        self.fit(X, y)
        self._train_y_ = y
        result = self.transform(X)
        self._train_y_ = None
        return result

    def transform(self, X):
        df = X.copy()
        df = self._fill_missing(df)
        df = self._ordinal_encode(df)
        df = self._apply_te(df)
        df = self._geo_features(df)
        df = self._feature_engineering(df)
        df = self._qol_features(df)
        df = self._drop_redundant(df)
        df = self._drop_cols(df)
        df = self._fix_skewness(df)

        cat_cols = df.select_dtypes(include=['object']).columns.tolist()
        for col in cat_cols:
            df[col] = df[col].fillna('Missing')

        # SHAP Top-N 変数選抜
        if self.selected_features is not None:
            keep = [c for c in self.selected_features if c in df.columns]
            df = df[keep]
            cat_cols = [c for c in cat_cols if c in keep]

        self.cat_feature_names_ = cat_cols
        self.cat_feature_indices_ = [df.columns.get_loc(c) for c in cat_cols]
        self.feature_names_ = df.columns.tolist()
        return df

    # --- 欠損値処理 ---
    def _fill_missing(self, df):
        for col in self.NA_MEANS_NONE:
            if col in df.columns:
                df[col] = df[col].fillna('None')
        for col in ['GarageYrBlt', 'GarageCars', 'GarageArea']:
            if col in df.columns:
                df[col] = df[col].fillna(0)
        for col in ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                     'BsmtFullBath', 'BsmtHalfBath']:
            if col in df.columns:
                df[col] = df[col].fillna(0)
        if 'LotFrontage' in df.columns:
            for idx in df[df['LotFrontage'].isnull()].index:
                nbhd = df.loc[idx, 'Neighborhood']
                med = self.lot_frontage_medians_.get(nbhd, self.lot_frontage_global_)
                df.loc[idx, 'LotFrontage'] = med
        df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
        num_cols = df.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if df[col].isnull().any() and col in self.num_medians_:
                df[col] = df[col].fillna(self.num_medians_[col])
        cat_cols = df.select_dtypes(include=['object']).columns
        for col in cat_cols:
            if df[col].isnull().any() and col in self.cat_modes_:
                df[col] = df[col].fillna(self.cat_modes_[col])
        return df

    def _ordinal_encode(self, df):
        for col, mapping in self.ORDINAL_MAP.items():
            if col in df.columns:
                df[col] = df[col].map(mapping).fillna(0).astype(int)
        return df

    # --- Target Encoding (廃止済み、互換性のため残す) ---
    def _fit_te(self, df, y):
        self.te_global_mean_ = float(y.mean()) if y is not None else None
        self.te_stats_ = {}

    def _apply_te(self, df):
        return df

    # --- 地理的特徴量 ---
    def _geo_features(self, df):
        """Neighborhood → Lat/Lon/Dist_to_Center/Dist_to_HighPrice_Center に変換"""
        if 'Neighborhood' not in df.columns:
            return df

        # 全体の中央値 (未知の Neighborhood 用フォールバック)
        all_lats = [c[0] for c in NEIGHBORHOOD_COORDS.values()]
        all_lons = [c[1] for c in NEIGHBORHOOD_COORDS.values()]
        fallback_lat = np.median(all_lats)
        fallback_lon = np.median(all_lons)

        df['Latitude'] = df['Neighborhood'].map(
            {k: v[0] for k, v in NEIGHBORHOOD_COORDS.items()}).fillna(fallback_lat)
        df['Longitude'] = df['Neighborhood'].map(
            {k: v[1] for k, v in NEIGHBORHOOD_COORDS.items()}).fillna(fallback_lon)

        # Ames 市中心からのユークリッド距離
        center_lat, center_lon = AMES_CENTER
        df['Dist_to_Center'] = np.sqrt(
            (df['Latitude'] - center_lat) ** 2 +
            (df['Longitude'] - center_lon) ** 2
        )

        # 高価格エリア中心からのユークリッド距離
        hp_lat, hp_lon = HIGH_PRICE_CENTER
        df['Dist_to_HighPrice_Center'] = np.sqrt(
            (df['Latitude'] - hp_lat) ** 2 +
            (df['Longitude'] - hp_lon) ** 2
        )
        return df

    # --- 特徴量エンジニアリング ---
    def _feature_engineering(self, df):
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath']
                           + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df['IsNew'] = (df['YearBuilt'] == df['YrSold']).astype(int)
        df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch']
                              + df['3SsnPorch'] + df['ScreenPorch'])
        df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
        df['HasRemod'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
        df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
        df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
        df['Has2ndFlr'] = (df['2ndFlrSF'] > 0).astype(int)
        df['HasPool'] = (df['PoolArea'] > 0).astype(int)
        if self.first_floor_median_ is not None:
            df['Is_SFL'] = ((df['2ndFlrSF'] == 0)
                            & (df['1stFlrSF'] >= self.first_floor_median_)).astype(int)
        df['Living_Space_Ratio'] = df['GrLivArea'] / df['LotArea'].clip(lower=1)
        df['Luxury_Space_Index'] = df['TotalSF'] * df['OverallQual']
        return df

    def _qol_features(self, df):
        df['QOL_Score'] = (
            (df['CentralAir'] == 1).astype(int)
            + (df['GarageArea'] > 0).astype(int)
            + (df['TotalBsmtSF'] > 0).astype(int)
            + (df['Fireplaces'] > 0).astype(int)
            + (df['Functional'] == 8).astype(int)
            + (df['KitchenQual'] >= 4).astype(int)
            + (df['ExterQual'] >= 4).astype(int)
            + (df['BsmtQual'] >= 4).astype(int)
            + (df['FullBath'] >= 2).astype(int)
        )
        return df

    def _drop_redundant(self, df):
        """合成特徴量の元変数を削除 (TotalSF, TotalBath, HouseAge に集約済み)"""
        drop = [c for c in REDUNDANT_COLS if c in df.columns]
        return df.drop(columns=drop)

    def _drop_cols(self, df):
        drop = [c for c in self.DROP_COLS if c in df.columns]
        return df.drop(columns=drop)

    def _fix_skewness(self, df):
        for feat in self.SKEW_FEATURES:
            if feat in df.columns:
                df[feat] = np.log1p(np.maximum(df[feat], 0))
        for feat in ['TotalSF', 'TotalPorchSF', 'Luxury_Space_Index']:
            if feat in df.columns:
                df[feat] = np.log1p(np.maximum(df[feat], 0))
        return df

print(f"HousePricesPreprocessor defined (Geo features: Lat/Lon/Dist_to_Center/Dist_to_HighPrice_Center, TE disabled).")
print(f"  SHAP_TOP_N={SHAP_TOP_N}, CLIP=[{CLIP_MIN}, {CLIP_MAX}]")
print(f"  AMES_CENTER={AMES_CENTER}")
print(f"  HIGH_PRICE_CENTER={HIGH_PRICE_CENTER}")

## Pipeline ラッパー
- **CatBoostPipeline**: cat_features を自動渡し (カテゴリカル=object のまま)
- **SklearnPipeline**: object 列を LabelEncode して数値化 (Ridge, GBR 等用)

In [ ]:
class CatBoostPipeline:
    """CatBoost用: cat_features を自動的に渡す"""

    def __init__(self, model, selected_features=None):
        self.preprocess = HousePricesPreprocessor(selected_features=selected_features)
        self.model = model

    def fit(self, X, y):
        X_t = self.preprocess.fit_transform(X, y)
        cat_idx = self.preprocess.cat_feature_indices_
        self.model.fit(X_t, y, cat_features=cat_idx)
        return self

    def predict(self, X):
        X_t = self.preprocess.transform(X)
        return self.model.predict(X_t)


class SklearnPipeline:
    """sklearn用: object 列を LabelEncode して数値化"""

    def __init__(self, model, selected_features=None):
        self.preprocess = HousePricesPreprocessor(selected_features=selected_features)
        self.model = model
        self.label_maps_ = {}

    def fit(self, X, y):
        X_t = self.preprocess.fit_transform(X, y)
        X_t = self._label_encode_fit(X_t)
        self.model.fit(X_t.values.astype(np.float64), y)
        return self

    def predict(self, X):
        X_t = self.preprocess.transform(X)
        X_t = self._label_encode_transform(X_t)
        return self.model.predict(X_t.values.astype(np.float64))

    def _label_encode_fit(self, df):
        self.label_maps_ = {}
        for col in df.select_dtypes(include=['object']).columns:
            vals = sorted(df[col].unique())
            self.label_maps_[col] = {v: i for i, v in enumerate(vals)}
            df[col] = df[col].map(self.label_maps_[col]).fillna(-1).astype(int)
        return df

    def _label_encode_transform(self, df):
        for col, mapping in self.label_maps_.items():
            if col in df.columns:
                df[col] = df[col].map(mapping).fillna(-1).astype(int)
        return df


def make_pipeline(model, use_catboost=False, selected_features=None):
    if use_catboost:
        return CatBoostPipeline(model, selected_features=selected_features)
    return SklearnPipeline(model, selected_features=selected_features)

print("CatBoostPipeline & SklearnPipeline defined (with selected_features support).")

## データ読み込み

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

# 外れ値除去 (GrLivArea > 4000 & 低価格)
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index
train = train.drop(outliers).reset_index(drop=True)
print(f'外れ値除去: {len(outliers)}件 → Train: {len(train)}行')

y = np.log1p(train['SalePrice'])
X = train.drop(columns=['SalePrice'])
X_test = test.copy()
test_ids = test['Id']

print(f'X: {X.shape}, X_test: {X_test.shape}')

## KNN 地理的価格特徴量 (OOF)
Neighborhood_te の代替: 地理的に近い K 物件の LogPrice 平均を OOF で算出し、X / X_test に注入。

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

def compute_knn_geo_price(X_train, y_train, X_test, n_neighbors=10, n_splits=5, seed=42):
    """
    KNN地理的価格特徴量: Lat/Lon を使い近隣 K 物件の LogPrice 平均を算出。
    訓練データは OOF (leak-free)、テストは全訓練データで予測。
    """
    # Neighborhood → Lat/Lon 変換
    def get_coords(df):
        all_lats = [c[0] for c in NEIGHBORHOOD_COORDS.values()]
        all_lons = [c[1] for c in NEIGHBORHOOD_COORDS.values()]
        fallback_lat, fallback_lon = np.median(all_lats), np.median(all_lons)
        lat = df['Neighborhood'].map({k: v[0] for k, v in NEIGHBORHOOD_COORDS.items()}).fillna(fallback_lat).values
        lon = df['Neighborhood'].map({k: v[1] for k, v in NEIGHBORHOOD_COORDS.items()}).fillna(fallback_lon).values
        return np.column_stack([lat, lon])

    coords_train = get_coords(X_train)
    coords_test = get_coords(X_test)
    y_arr = y_train.values if hasattr(y_train, 'values') else np.array(y_train)

    # OOF predictions for train
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_knn = np.zeros(len(X_train))

    for train_idx, val_idx in kf.split(coords_train):
        knn = KNeighborsRegressor(n_neighbors=n_neighbors, weights='distance', metric='euclidean')
        knn.fit(coords_train[train_idx], y_arr[train_idx])
        oof_knn[val_idx] = knn.predict(coords_train[val_idx])

    # Test predictions (full train)
    knn_full = KNeighborsRegressor(n_neighbors=n_neighbors, weights='distance', metric='euclidean')
    knn_full.fit(coords_train, y_arr)
    test_knn = knn_full.predict(coords_test)

    # 評価
    knn_rmsle = np.sqrt(mean_squared_error(y_arr, oof_knn))
    print(f'KNN Geo Price (K={n_neighbors}): OOF RMSLE = {knn_rmsle:.5f}')

    return oof_knn, test_knn

# KNN 特徴量を算出し、X / X_test に列として追加
knn_train, knn_test = compute_knn_geo_price(X, y, X_test, n_neighbors=10)
X = X.copy()
X_test = X_test.copy()
X['KNN_Geo_Price'] = knn_train
X_test['KNN_Geo_Price'] = knn_test
print(f'✅ KNN_Geo_Price を X, X_test に追加 (X: {X.shape}, X_test: {X_test.shape})')

## SHAP ベース変数選抜 (Top 40)

In [ ]:
# SHAP importance で Top-N 変数を決定 (LightGBM feature_importances_ で高速近似)
_prep = HousePricesPreprocessor()  # selected_features=None で全変数
_X_all = _prep.fit_transform(X, y)

# カテゴリカルを LabelEncode (LightGBM用)
_label_maps = {}
for col in _X_all.select_dtypes(include=['object']).columns:
    vals = sorted(_X_all[col].unique())
    _label_maps[col] = {v: i for i, v in enumerate(vals)}
    _X_all[col] = _X_all[col].map(_label_maps[col]).fillna(-1).astype(int)

_lgb_fs = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.01, max_depth=3,
    num_leaves=7, min_child_samples=50, subsample=0.8,
    colsample_bytree=0.8, random_state=42, verbose=-1)
_lgb_fs.fit(_X_all.values.astype(np.float64), y)

_importances = pd.Series(_lgb_fs.feature_importances_, index=_prep.feature_names_)
_importances = _importances.sort_values(ascending=False)

# Neighborhood_te を明示的に除外 (TE廃止の安全策)
_importances = _importances.drop(labels=[c for c in _importances.index if c.endswith('_te')], errors='ignore')

SELECTED_FEATURES = _importances.head(SHAP_TOP_N).index.tolist()

# Neighborhood を強制的に含める (CatBoost のカテゴリカル処理用)
FORCE_INCLUDE = ['Neighborhood']
for feat in FORCE_INCLUDE:
    if feat in _prep.feature_names_ and feat not in SELECTED_FEATURES:
        SELECTED_FEATURES.append(feat)
        print(f"   ⚡ {feat} を強制追加 (SHAP Top-{SHAP_TOP_N} 外)")

print(f"✅ SHAP Top {SHAP_TOP_N} + forced features → 選抜: {len(SELECTED_FEATURES)} 変数")
print(f"   全変数: {len(_prep.feature_names_)} → 選抜: {len(SELECTED_FEATURES)}")
# Geo features の有無を確認
_geo_in = [f for f in ['Latitude', 'Longitude', 'Dist_to_Center', 'Dist_to_HighPrice_Center', 'KNN_Geo_Price', 'Neighborhood'] if f in SELECTED_FEATURES]
print(f"   Geo/Location features: {_geo_in}")
for i, f in enumerate(SELECTED_FEATURES):
    imp_val = _importances.get(f, 0)
    print(f"   {i+1:2d}. {f} (importance: {imp_val:.0f})")

# cleanup
del _prep, _X_all, _lgb_fs, _label_maps, _importances

## Stage 1: Baseline Anchor (Ridge on 3 core features)
**Residual Learning**: 住宅価格の「基本的な価値」と「細かな付加価値」を分離して学習する2段階構造。
- **Stage 1**: TotalSF, OverallQual, LotArea の3変数で Ridge 回帰 → ベースライン予測
- **Stage 2**: 7モデル OOF スタッキングで「残差」を学習 (HouseAge 含む全特徴量)
- **最終予測**: baseline_pred + residual_pred

In [ ]:
from sklearn.preprocessing import StandardScaler

BASELINE_FEATURES = ['TotalSF', 'OverallQual', 'LotArea']

def compute_baseline_oof(X, y, X_test, n_splits=5, seed=42, alpha=10.0):
    """
    Stage 1: Ridge baseline on 3 core features with OOF predictions.
    Returns OOF predictions for train (leak-free) and averaged predictions for test.
    """
    # Preprocess to get engineered features (TotalSF etc.)
    prep_base = HousePricesPreprocessor(selected_features=None)
    X_all = prep_base.fit_transform(X, y)
    X_test_all = prep_base.transform(X_test)

    # Extract baseline features only
    X_train_base = X_all[BASELINE_FEATURES].values.astype(np.float64)
    X_test_base = X_test_all[BASELINE_FEATURES].values.astype(np.float64)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # OOF predictions (leak-free)
    oof_base_pred = np.zeros(len(X))
    test_base_preds = np.zeros((len(X_test), n_splits))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_base)):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train_base[train_idx])
        X_va = scaler.transform(X_train_base[val_idx])
        X_te = scaler.transform(X_test_base)

        ridge = Ridge(alpha=alpha)
        ridge.fit(X_tr, y.values[train_idx])

        oof_base_pred[val_idx] = ridge.predict(X_va)
        test_base_preds[:, fold] = ridge.predict(X_te)

    test_base_pred = test_base_preds.mean(axis=1)

    # Stage 1 evaluation
    stage1_rmsle = np.sqrt(mean_squared_error(y, oof_base_pred))
    print(f'Stage 1 (Ridge baseline on {BASELINE_FEATURES}):')
    print(f'  RMSLE = {stage1_rmsle:.5f}')
    print(f'  Alpha = {alpha}')

    return oof_base_pred, test_base_pred, stage1_rmsle

# Compute baseline for default seed (for CV evaluation)
print('=' * 60)
print(f'Stage 1: Baseline Anchor (Ridge on {len(BASELINE_FEATURES)} core features)')
print('=' * 60)
oof_base_pred_default, test_base_pred_default, stage1_score = compute_baseline_oof(X, y, X_test, seed=42)

# Residuals for default seed
train_residuals_default = y - oof_base_pred_default
print(f'\n  Residual stats: mean={train_residuals_default.mean():.5f}, std={train_residuals_default.std():.5f}')

## モデル定義 & CV評価

In [ ]:
def _catboost_gpu_available():
    """CatBoost GPU が使えるか実際に試行"""
    if not IS_COLAB:
        return False
    try:
        from catboost import CatBoostRegressor as _CB
        _cb = _CB(iterations=1, task_type='GPU', devices='0', verbose=0)
        _cb.fit([[0],[1]], [0,1])
        return True
    except Exception:
        return False

USE_CATBOOST_GPU = _catboost_gpu_available()
print(f"CatBoost GPU: {'available' if USE_CATBOOST_GPU else 'not available (using CPU)'}")

def get_models(seed=42):
    """全モデル定義 (XGB/LGB/CB 含む)"""
    cb_params = dict(
        iterations=5000,
        learning_rate=0.05, depth=6,
        l2_leaf_reg=10, od_type='Iter', od_wait=100,
        random_seed=seed, verbose=0,
    )
    if USE_CATBOOST_GPU:
        cb_params['task_type'] = 'GPU'
        cb_params['devices'] = '0'

    return {
        'Ridge': Ridge(alpha=5.0),
        'Lasso': Lasso(alpha=0.0003, max_iter=10000),
        'ElasticNet': ElasticNet(alpha=0.0005, l1_ratio=0.7, max_iter=10000),
        'GBR': GradientBoostingRegressor(
            n_estimators=3000, learning_rate=0.05, max_depth=4,
            min_samples_split=15, min_samples_leaf=10, max_features='sqrt',
            loss='huber', subsample=0.75, random_state=seed),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=5000, learning_rate=0.005, max_depth=5,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
            reg_lambda=1.0, random_state=seed, verbosity=0),
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=5000, learning_rate=0.01, max_depth=3,
            num_leaves=7, min_child_samples=50, subsample=0.8,
            colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
            random_state=seed, verbose=-1),
        'CatBoost': CatBoostRegressor(**cb_params),
    }

CATBOOST_MODELS = {'CatBoost'}
print("Models (変更後):")
print("  XGBoost:  n_est=5000, lr=0.005, depth=5  (↑from 3)")
print("  LightGBM: n_est=5000, lr=0.01,  depth=3, leaves=7  (lr↑from 0.008)")
print("  CatBoost: iter=5000,  lr=0.05,  depth=6  (lr↑from 0.02)")

In [ ]:
RMSLE_THRESHOLD = 0.108

def evaluate_models(X, y, n_splits=5, seed=42):
    """KFold CVで複数モデルを評価 (RMSLE = RMSE on log1p target)"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = get_models(seed)

    results = {}
    for name, model in models.items():
        is_cb = name in CATBOOST_MODELS
        rmse_list = []
        for train_idx, val_idx in kf.split(X):
            pipe = make_pipeline(model, use_catboost=is_cb, selected_features=SELECTED_FEATURES)
            pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred = pipe.predict(X.iloc[val_idx])
            rmse_list.append(np.sqrt(mean_squared_error(y.iloc[val_idx], pred)))
        rmse_scores = np.array(rmse_list)
        mean_rmse = rmse_scores.mean()
        results[name] = {
            'mean': mean_rmse,
            'std': rmse_scores.std(),
            'scores': rmse_scores,
        }
        status = '✅' if mean_rmse < RMSLE_THRESHOLD else '  '
        print(f'{status} {name:12s}: RMSLE = {mean_rmse:.5f} ± {rmse_scores.std():.5f}')

    # サマリー
    print(f'\n--- Threshold: {RMSLE_THRESHOLD} ---')
    below = [n for n, r in results.items() if r['mean'] < RMSLE_THRESHOLD]
    if below:
        print(f'✅ {len(below)} model(s) below threshold: {", ".join(below)}')
    else:
        best_val = min(r['mean'] for r in results.values())
        print(f'⚠️  No model below threshold (best: {best_val:.5f}, gap: +{best_val - RMSLE_THRESHOLD:.5f})')

    return results

print('=' * 60)
print(f'モデル比較 (5-fold CV, RMSLE) - Top {SHAP_TOP_N} features [直接予測]')
print('=' * 60)
results = evaluate_models(X, y)

best_model = min(results, key=lambda k: results[k]['mean'])
print(f'\nベストモデル: {best_model} (RMSLE={results[best_model]["mean"]:.5f})')

# --- Residual Learning での CV 評価 ---
print(f'\n{"=" * 60}')
print(f'モデル比較 (5-fold CV, RMSLE) - Top {SHAP_TOP_N} features [Residual Learning]')
print(f'{"=" * 60}')

def evaluate_models_residual(X, y, oof_base_pred, n_splits=5, seed=42):
    """Residual Learning: Stage 2 のモデルを残差で学習し、Stage 1 + Stage 2 の合計 RMSLE を評価"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = get_models(seed)
    residuals = y - oof_base_pred  # 学習ターゲット: 残差

    results = {}
    for name, model in models.items():
        is_cb = name in CATBOOST_MODELS
        rmse_list = []
        for train_idx, val_idx in kf.split(X):
            pipe = make_pipeline(model, use_catboost=is_cb, selected_features=SELECTED_FEATURES)
            # 残差をターゲットとして学習
            pipe.fit(X.iloc[train_idx], residuals.iloc[train_idx])
            residual_pred = pipe.predict(X.iloc[val_idx])
            # 最終予測 = baseline + residual
            final_pred = oof_base_pred[val_idx] + residual_pred
            rmse_list.append(np.sqrt(mean_squared_error(y.iloc[val_idx], final_pred)))
        rmse_scores = np.array(rmse_list)
        mean_rmse = rmse_scores.mean()
        results[name] = {
            'mean': mean_rmse,
            'std': rmse_scores.std(),
            'scores': rmse_scores,
        }
        status = '✅' if mean_rmse < RMSLE_THRESHOLD else '  '
        print(f'{status} {name:12s}: RMSLE = {mean_rmse:.5f} ± {rmse_scores.std():.5f}')

    return results

results_residual = evaluate_models_residual(X, y, oof_base_pred_default)
best_residual = min(results_residual, key=lambda k: results_residual[k]['mean'])
print(f'\nベストモデル (Residual): {best_residual} (RMSLE={results_residual[best_residual]["mean"]:.5f})')

## 単体ベストモデルで予測

In [ ]:
import os

# Colab: 絶対パスで固定 / Local: プロジェクトルート基準
if IS_COLAB:
    SUBMISSION_DIR = '/content/submissions'
    FIGURES_DIR = '/content/figures'
else:
    from pathlib import Path as _Path
    _project_root = _Path(DATA_DIR).resolve().parent if _Path(DATA_DIR).name == 'data' else _Path('.')
    SUBMISSION_DIR = str(_project_root / 'submissions')
    FIGURES_DIR = str(_project_root / 'figures')

os.makedirs(SUBMISSION_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"SUBMISSION_DIR: {SUBMISSION_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")

def train_and_predict(X, y, X_test, test_ids, model_name='GBR', seed=42):
    """フル学習 → テスト予測 → submission.csv"""
    models = get_models(seed)
    model = models[model_name]
    is_cb = model_name in CATBOOST_MODELS
    pipe = make_pipeline(model, use_catboost=is_cb, selected_features=SELECTED_FEATURES)

    pipe.fit(X, y)
    preds_log = pipe.predict(X_test)
    preds = np.expm1(preds_log)
    preds = np.maximum(preds, 0)
    preds = np.clip(preds, CLIP_MIN, CLIP_MAX)

    submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
    filename = f'{SUBMISSION_DIR}/submission_{model_name.lower()}.csv'
    submission.to_csv(filename, index=False)
    print(f'\n{filename} を作成しました ({len(submission)}行)')
    print(f'SalePrice: mean={preds.mean():.0f}, median={np.median(preds):.0f}')

    return submission

train_and_predict(X, y, X_test, test_ids, model_name=best_model)

## OOF スタッキング

In [ ]:
def stacking_predict(X, y, X_test, test_ids, seed=42):
    """Residual Learning OOFスタッキング:
    Stage 1: Ridge baseline (3 features) → OOF baseline predictions
    Stage 2: 7-model OOF stacking on residuals → Ridge meta-model
    Final: baseline + residual predictions
    """
    # --- Stage 1: Baseline ---
    oof_base_pred, test_base_pred, stage1_rmsle = compute_baseline_oof(
        X, y, X_test, n_splits=5, seed=seed)
    train_residuals = y - oof_base_pred

    # --- Stage 2: Residual Stacking ---
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    base_models = get_models(seed)

    n_train = len(X)
    n_test = len(X_test)
    n_models = len(base_models)

    oof_preds = np.zeros((n_train, n_models))
    test_preds = np.zeros((n_test, n_models))

    for i, (name, model) in enumerate(base_models.items()):
        print(f'  {name}...', end='', flush=True)
        test_preds_fold = np.zeros((n_test, 5))
        is_cb = name in CATBOOST_MODELS

        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            pipe = make_pipeline(model, use_catboost=is_cb, selected_features=SELECTED_FEATURES)
            # 残差をターゲットとして学習
            pipe.fit(X.iloc[train_idx], train_residuals.iloc[train_idx])
            oof_preds[val_idx, i] = pipe.predict(X.iloc[val_idx])
            test_preds_fold[:, fold] = pipe.predict(X_test)

        test_preds[:, i] = test_preds_fold.mean(axis=1)
        # OOF 残差予測の精度 (残差そのもののRMSE)
        residual_rmse = np.sqrt(mean_squared_error(train_residuals, oof_preds[:, i]))
        # 合計 RMSLE (baseline + residual vs y)
        combined_rmse = np.sqrt(mean_squared_error(y, oof_base_pred + oof_preds[:, i]))
        print(f' residual_RMSE={residual_rmse:.5f}, combined_RMSLE={combined_rmse:.5f}')

    # メタモデル (残差の予測を統合)
    meta = Ridge(alpha=1.0)
    meta.fit(oof_preds, train_residuals)
    meta_oof_pred = meta.predict(oof_preds)
    meta_test_pred = meta.predict(test_preds)

    # Stage 2 合計 CV スコア
    final_oof = oof_base_pred + meta_oof_pred
    stage2_rmsle = np.sqrt(mean_squared_error(y, final_oof))

    # メタモデルの CV (残差ベース)
    meta_cv = np.sqrt(-cross_val_score(
        Ridge(alpha=1.0), oof_preds, train_residuals,
        cv=KFold(5, shuffle=True, random_state=seed),
        scoring='neg_mean_squared_error'))

    print(f'\n  Stage 1 RMSLE (Ridge baseline): {stage1_rmsle:.5f}')
    print(f'  Stage 2 RMSLE (baseline + stack): {stage2_rmsle:.5f}')
    print(f'  Meta residual CV RMSE: {meta_cv.mean():.5f} ± {meta_cv.std():.5f}')
    print(f'  Meta weights: {dict(zip(base_models.keys(), meta.coef_.round(3)))}')

    # --- Final Integration ---
    final_log_preds = test_base_pred + meta_test_pred
    preds = np.expm1(final_log_preds)
    preds = np.maximum(preds, 0)
    preds = np.clip(preds, CLIP_MIN, CLIP_MAX)

    submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
    submission.to_csv(f'{SUBMISSION_DIR}/submission_stack.csv', index=False)
    print(f'\n{SUBMISSION_DIR}/submission_stack.csv を作成しました ({len(submission)}行)')
    print(f'SalePrice: mean={preds.mean():.0f}, median={np.median(preds):.0f}')

    return submission, stage2_rmsle

print('=' * 60)
print(f'Residual Learning スタッキング (Top {SHAP_TOP_N} features)')
print('=' * 60)
stacking_predict(X, y, X_test, test_ids)

## SHAP 値 上位20変数

In [ ]:
!pip install -q shap
import shap
import matplotlib.pyplot as plt

# 選抜済み変数で LightGBM → SHAP
prep = HousePricesPreprocessor(selected_features=SELECTED_FEATURES)
X_processed = prep.fit_transform(X, y)

print(f"✅ 変数数: {X_processed.shape[1]} (Top {SHAP_TOP_N})")
print(f"   変数: {prep.feature_names_}")

# カテゴリカル列を LabelEncode (SHAP の TreeExplainer 用)
label_maps = {}
for col in X_processed.select_dtypes(include=['object']).columns:
    vals = sorted(X_processed[col].unique())
    label_maps[col] = {v: i for i, v in enumerate(vals)}
    X_processed[col] = X_processed[col].map(label_maps[col]).fillna(-1).astype(int)

X_np = X_processed.values.astype(np.float64)
feature_names = prep.feature_names_

lgb_model = lgb.LGBMRegressor(
    n_estimators=4000, learning_rate=0.01, max_depth=3,
    num_leaves=7, min_child_samples=50, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbose=-1)
lgb_model.fit(X_np, y)

explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_np)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_np, feature_names=feature_names,
                  max_display=SHAP_TOP_N, show=False)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/shap_top{SHAP_TOP_N}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"{FIGURES_DIR}/shap_top{SHAP_TOP_N}.png saved")

## マルチシード スタッキング (Seed Averaging)

In [ ]:
# === 1シード評価モード (seed=42) ===
# 方向性確認後、マルチシードに戻す
EVAL_SEEDS = [42]  # 本番: [42, 123, 456]

print('=' * 60)
print(f'Residual Learning スタッキング (seeds: {EVAL_SEEDS}) - {len(SELECTED_FEATURES)} features')
print(f'  Stage 1: {BASELINE_FEATURES}')
print(f'  XGB: lr=0.005/5000/depth=5, LGB: lr=0.01/5000, CB: lr=0.05/5000')
print('=' * 60)
multi_preds = []
cv_scores = []
for seed in EVAL_SEEDS:
    print(f'\n--- seed={seed} ---')
    sub, cv = stacking_predict(X, y, X_test, test_ids, seed=seed)
    multi_preds.append(sub['SalePrice'].values)
    cv_scores.append(cv)

avg_preds = np.mean(multi_preds, axis=0)
avg_preds = np.clip(avg_preds, CLIP_MIN, CLIP_MAX)
submission_multi = pd.DataFrame({'Id': test_ids, 'SalePrice': avg_preds})

# 通常パス + 固定パス (Colab)
submission_multi.to_csv(f'{SUBMISSION_DIR}/submission_multiseed.csv', index=False)
if IS_COLAB:
    FINAL_SUBMISSION = '/content/final_submission.csv'
    submission_multi.to_csv(FINAL_SUBMISSION, index=False)
    print(f'\n✅ {FINAL_SUBMISSION} を作成しました (固定パス)')

print(f'{SUBMISSION_DIR}/submission_multiseed.csv を作成しました')
print(f'SalePrice: mean={avg_preds.mean():.0f}, median={np.median(avg_preds):.0f}')

print(f'\n{"=" * 60}')
print(f'CV サマリー (seeds: {EVAL_SEEDS})')
print(f'  各シード CV (Stage1+Stage2): {[f"{s:.5f}" for s in cv_scores]}')
if len(cv_scores) > 1:
    print(f'  平均: {np.mean(cv_scores):.5f} ± {np.std(cv_scores):.5f}')
print(f'  🎯 Target: < 0.111')
print(f'{"=" * 60}')

## 提出ファイルダウンロード (Colab用)

In [ ]:
if IS_COLAB:
    import shutil, glob
    # Google Drive に成果物を自動同期
    DRIVE_OUTPUT = '/content/drive/MyDrive/kaggle-output/house-prices'
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    os.makedirs(f'{DRIVE_OUTPUT}/submissions', exist_ok=True)
    os.makedirs(f'{DRIVE_OUTPUT}/figures', exist_ok=True)

    # submissions
    for f in glob.glob(f'{SUBMISSION_DIR}/submission_*.csv'):
        shutil.copy2(f, f'{DRIVE_OUTPUT}/submissions/')
    # final_submission も同期
    if os.path.exists('/content/final_submission.csv'):
        shutil.copy2('/content/final_submission.csv', f'{DRIVE_OUTPUT}/')
    print(f"✅ Submissions → {DRIVE_OUTPUT}/submissions/")

    # figures
    for f in glob.glob(f'{FIGURES_DIR}/*.png'):
        shutil.copy2(f, f'{DRIVE_OUTPUT}/figures/')
    print(f"✅ Figures → {DRIVE_OUTPUT}/figures/")

    # ブラウザダウンロードも実行
    from google.colab import files
    for f in glob.glob(f'{SUBMISSION_DIR}/submission_*.csv'):
        files.download(f)
else:
    print(f"Local: submission files saved in {SUBMISSION_DIR}/")
    !ls -la {SUBMISSION_DIR}/submission_*.csv

## Kaggle 自動提出 + スコア取得

In [ ]:
import requests, time, json
from pathlib import Path

# --- Kaggle トークン取得 ---
_kaggle_token = None

# 1. ~/.kaggle/kaggle.json
_kj = Path.home() / '.kaggle' / 'kaggle.json'
if _kj.exists():
    _kaggle_token = json.loads(_kj.read_text()).get('key', '')

# 2. Colab Secrets
if not _kaggle_token and IS_COLAB:
    try:
        from google.colab import userdata
        _raw = userdata.get('KAGGLE_TOKEN')
        if _raw:
            try:
                _kaggle_token = json.loads(_raw).get('key', _raw)
            except (json.JSONDecodeError, TypeError):
                _kaggle_token = _raw
    except Exception:
        pass

# 3. Google Drive
if not _kaggle_token and IS_COLAB:
    _dkj = Path('/content/drive/MyDrive/.kaggle/kaggle.json')
    if _dkj.exists():
        _kaggle_token = json.loads(_dkj.read_text()).get('key', '')

if not _kaggle_token:
    print("⚠️ Kaggle トークンが見つかりません。手動で提出してください。")
else:
    COMPETITION = 'house-prices-advanced-regression-techniques'
    SUBMIT_FILE = f'{SUBMISSION_DIR}/submission_multiseed.csv'
    SUBMIT_DESC = f'Geo+Dist_to_Center (no TE) + Residual Top-{SHAP_TOP_N} multiseed'

    _headers = {'Authorization': f'Bearer {_kaggle_token}', 'Content-Type': 'application/json'}
    _file_size = os.path.getsize(SUBMIT_FILE)

    try:
        # Step 1: StartSubmissionUpload
        print(f'提出: {SUBMIT_FILE} ({_file_size:,} bytes)')
        print('Step 1/3: アップロード開始...', end='', flush=True)
        r = requests.post(
            'https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload',
            headers=_headers,
            json={'competitionName': COMPETITION, 'fileName': 'submission_multiseed.csv', 'contentLength': _file_size})
        if not r.ok:
            print(f'\n❌ Step 1 failed: HTTP {r.status_code}\n{r.text[:1000]}')
            raise Exception('Step 1 failed')
        _resp = r.json()
        print(f' OK')
        _blob_token = _resp.get('token', _resp.get('blobToken', ''))
        _upload_url = _resp.get('createUrl', _resp.get('uploadUrl', ''))

        # Step 2: PUT to GCS
        print('Step 2/3: ファイルアップロード...', end='', flush=True)
        with open(SUBMIT_FILE, 'rb') as f:
            r = requests.put(_upload_url, headers={'Content-Type': 'text/csv'}, data=f)
        if not r.ok:
            print(f'\n❌ Step 2 failed: HTTP {r.status_code}\n{r.text[:1000]}')
            raise Exception('Step 2 failed')
        print(' OK')

        # Step 3: CreateSubmission (blobToken as singular string)
        print('Step 3/3: 提出作成...', flush=True)
        _submit_body = {
            'competitionName': COMPETITION,
            'blobToken': _blob_token,
            'submissionDescription': SUBMIT_DESC
        }
        r = requests.post(
            'https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission',
            headers=_headers,
            json=_submit_body)
        print(f'  Response: HTTP {r.status_code}')
        if r.text:
            print(f'  Body: {r.text[:500]}')
        if not r.ok:
            # Fallback: try array format
            print('  Retrying with blobFileTokens array format...')
            _submit_body2 = {
                'competitionName': COMPETITION,
                'blobFileTokens': [_blob_token],
                'submissionDescription': SUBMIT_DESC
            }
            r = requests.post(
                'https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission',
                headers=_headers,
                json=_submit_body2)
            print(f'  Retry Response: HTTP {r.status_code}')
            if r.text:
                print(f'  Body: {r.text[:500]}')
            if not r.ok:
                raise Exception(f'Step 3 failed: HTTP {r.status_code}')
        print(' OK')

        # スコア待機 (最大5分)
        print(f'\nスコア待機中 (最大300秒)...', flush=True)
        _start = time.time()
        _score = None
        while time.time() - _start < 300:
            time.sleep(10)
            r = requests.post(
                'https://api.kaggle.com/v1/competitions.CompetitionApiService/ListSubmissions',
                headers=_headers,
                json={'competitionName': COMPETITION, 'pageSize': 1})
            r.raise_for_status()
            _subs = r.json().get('submissions', [])
            if _subs:
                _latest = _subs[0]
                _status = _latest.get('status', '')
                _score = _latest.get('publicScore', '')
                _elapsed = int(time.time() - _start)
                if _status == 'COMPLETE' and _score:
                    print(f'\n{"=" * 50}')
                    print(f'  ✅ Public Score: {_score}')
                    print(f'  File: {_latest.get("fileName", "")}')
                    print(f'  Desc: {_latest.get("description", "")}')
                    print(f'  ({_elapsed}秒で確定)')
                    print(f'{"=" * 50}')
                    break
                elif _status == 'ERROR':
                    print(f'\n❌ 提出エラー: {_latest}')
                    break
                else:
                    print(f'  ... {_elapsed}秒経過 (status: {_status})', flush=True)
        else:
            print(f'\n⏰ タイムアウト (300秒). Kaggle Web で確認してください。')

    except Exception as e:
        print(f'\n⚠️ 自動提出に失敗: {e}')
        print(f'手動で提出してください: {SUBMIT_FILE}')
    finally:
        del _kaggle_token, _headers, _file_size